In [0]:
%sql
-- Daily Performance Summary
WITH ranked AS (
  SELECT
    order_date,
    total_sales,
    total_profit,
    profit_margin,
    ROW_NUMBER() OVER (ORDER BY order_date DESC) AS rn
  FROM superstore_catalog.dev_semantic_layer.metrics_daily_kpi
),
kpi AS (
  SELECT
    MAX(CASE WHEN rn = 1 THEN total_sales END) AS revenue,
    MAX(CASE WHEN rn = 2 THEN total_sales END) AS prev_revenue,

    MAX(CASE WHEN rn = 1 THEN total_profit END) AS profit,
    MAX(CASE WHEN rn = 2 THEN total_profit END) AS prev_profit,

    MAX(CASE WHEN rn = 1 THEN profit_margin END) AS margin,
    MAX(CASE WHEN rn = 2 THEN profit_margin END) AS prev_margin
  FROM ranked
),
calc AS (
  SELECT
    revenue,
    (revenue - prev_revenue)/prev_revenue * 100 AS revenue_change,

    profit,
    (profit - prev_profit)/prev_profit * 100 AS profit_change,

    margin,
    (margin - prev_margin)/prev_margin * 100 AS margin_change
  FROM kpi
)

SELECT
  -- Revenue
  CONCAT('$', ROUND(revenue/1000000,2), 'M') AS revenue,
  CONCAT(
    CASE WHEN revenue_change >= 0 THEN '▲ ' ELSE '▼ ' END,
    ROUND(ABS(revenue_change),2), '%'
  ) AS revenue_change,

  -- Profit
  CONCAT('$', ROUND(profit/1000,2), 'K') AS profit,
  CONCAT(
    CASE WHEN profit_change >= 0 THEN '▲ ' ELSE '▼ ' END,
    ROUND(ABS(profit_change),2), '%'
  ) AS profit_change,

  -- Margin
  CONCAT(ROUND(margin*100,2), '%') AS margin,
  CONCAT(
    CASE WHEN margin_change >= 0 THEN '▲ ' ELSE '▼ ' END,
    ROUND(ABS(margin_change),2), '%'
  ) AS margin_change

FROM calc;

In [0]:
%sql
-- Combined bar/line chart for daily total_sales, total_profit and profit_margin
SELECT
    order_date,
    total_sales,
    total_profit,
    profit_margin
FROM superstore_catalog.dev_semantic_layer.metrics_daily_kpi
ORDER BY order_date;

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- ===========================================
-- 7-Day Rolling Averages for Daily Sales KPIs
-- Ready for Dashboard Line/Area Charts
-- ===========================================
WITH rolling AS (
  SELECT
    order_date,
    
    -- 7-day rolling average metrics
    AVG(total_sales) OVER (ORDER BY order_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)/1000000 AS sales_7d_avg_million,
    AVG(total_profit) OVER (ORDER BY order_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)/1000 AS profit_7d_avg_thousand,
    AVG(profit_margin) OVER (ORDER BY order_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)*100 AS margin_7d_avg_pct
  FROM superstore_catalog.dev_semantic_layer.metrics_daily_kpi
)

SELECT
    order_date AS Date,
    
    -- Numeric columns for line/area charts
    sales_7d_avg_million AS sales_7d_avg,
    profit_7d_avg_thousand AS profit_7d_avg,
    margin_7d_avg_pct AS margin_7d_avg,
    
    -- Optional display columns for KPI cards
    CONCAT('$', ROUND(sales_7d_avg_million,2), 'M') AS sales_7d_display,
    CONCAT('$', ROUND(profit_7d_avg_thousand,2), 'K') AS profit_7d_display,
    CONCAT(ROUND(margin_7d_avg_pct,2), '%') AS margin_7d_display
FROM rolling
ORDER BY order_date;

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- KPI tile for top 10 product contribution to sales
SELECT
    metric_date,
    metric_value AS top_10_product_sales_pct
FROM superstore_catalog.dev_semantic_layer.metrics_business_kpi
WHERE metric_name = 'top_10_product_sales_pct'
ORDER BY metric_date;

In [0]:
%sql
-- KPI tiles: new customer growth and repeat customer rate
SELECT
    metric_date,
    metric_name,
    metric_value
FROM superstore_catalog.dev_semantic_layer.metrics_business_kpi
WHERE metric_name IN ('customer_growth_rate', 'repeat_customer_rate')
ORDER BY metric_date;